# TikTok Claims Classification Project

**Organization:** TikTok (pedagogical scenario)  
**Goal:** Build an ML model to classify TikTok videos as "claims" or "opinions" to reduce the content moderation backlog  
**Dataset:** 19,382 videos x 12 variables  
**Methodology:** PACE Framework (Plan → Analyze → Construct → Execute)  
**Tools:** Python (pandas, NumPy, scipy, scikit-learn, XGBoost, matplotlib, seaborn)  
**Champion Model:** Random Forest — 99.5% accuracy, 99.5% recall

---
## Course 1: Data Foundations (PACE: Plan)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

# Load data
df = pd.read_csv('tiktok_dataset.csv')

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("Column information:")
print(df.info())
print("\nDescriptive statistics:")
df.describe()

In [ ]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nRows with missing values: {df[df.isnull().any(axis=1)].shape[0]}")

# Drop missing values (298 rows — systematic pattern)
df_clean = df.dropna()
print(f"\nRows after dropping NAs: {len(df_clean)}")
print(f"Rows dropped: {len(df) - len(df_clean)}")

In [ ]:
# Value counts for categorical variables
print("Claim status distribution:")
print(df_clean['claim_status'].value_counts())
print("\nVerified status distribution:")
print(df_clean['verified_status'].value_counts())
print("\nAuthor ban status distribution:")
print(df_clean['author_ban_status'].value_counts())

In [ ]:
# Engagement by claim status
engagement_cols = ['video_view_count', 'video_like_count', 'video_share_count',
                   'video_download_count', 'video_comment_count']

print("Median engagement metrics by claim status:")
for col in engagement_cols:
    print(f"\n{col}:")
    print(df_clean.groupby('claim_status')[col].median())

---
## Course 2: Exploratory Data Analysis (PACE: Analyze — Part 1)

In [ ]:
# Reload cleaned data for consistent state
df = pd.read_csv('tiktok_dataset.csv')
df = df.dropna()

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

engagement_cols = ['video_view_count', 'video_like_count', 'video_share_count',
                   'video_download_count', 'video_comment_count']

In [ ]:
# Box plots by claim status (log scale)
for col in engagement_cols:
    plt.figure()
    sns.boxplot(data=df, x='claim_status', y=col)
    plt.yscale('log')
    plt.title(f'{col} by Claim Status (Log Scale)')
    plt.show()

In [ ]:
# Overlapping histograms (claims vs opinions)
plt.figure(figsize=(15, 10))
for i, col in enumerate(engagement_cols, 1):
    plt.subplot(2, 3, i)
    claims = df[df['claim_status'] == 'claim'][col]
    opinions = df[df['claim_status'] == 'opinion'][col]
    plt.hist(claims, bins=30, alpha=0.6, label='Claims', color='blue', edgecolor='black')
    plt.hist(opinions, bins=30, alpha=0.6, label='Opinions', color='orange', edgecolor='black')
    plt.title(f'{col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.legend()
    plt.yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Count plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x='claim_status', ax=axes[0])
axes[0].set_title('Count of Claims vs Opinions')

sns.countplot(data=df, x='claim_status', hue='author_ban_status', ax=axes[1])
axes[1].set_title('Claims vs Opinions by Author Ban Status')

sns.countplot(data=df, x='verified_status', ax=axes[2])
axes[2].set_title('Verified vs Not Verified Accounts')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots (log scale)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, y_col, title in zip(axes,
    ['video_like_count', 'video_comment_count', 'video_share_count'],
    ['Views vs Likes', 'Views vs Comments', 'Views vs Shares']):
    sns.scatterplot(data=df, x='video_view_count', y=y_col,
                    hue='claim_status', alpha=0.5, ax=ax)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(f'{title} by Claim Status (Log Scale)')

plt.tight_layout()
plt.show()

In [ ]:
# Engagement rate features
df['likes_per_view'] = df['video_like_count'] / df['video_view_count']
df['comments_per_view'] = df['video_comment_count'] / df['video_view_count']
df['shares_per_view'] = df['video_share_count'] / df['video_view_count']

print("Engagement rates by claim status:")
print(df.groupby('claim_status')[['likes_per_view', 'comments_per_view', 'shares_per_view']].mean())

---
## Course 3: Statistical Analysis (PACE: Analyze — Part 2)

**Null Hypothesis (H0):** No difference in mean video view counts between verified and not verified accounts.  
**Alternative Hypothesis (Ha):** There is a difference.  
**Significance level:** 0.05

In [ ]:
from scipy import stats

# Reload clean data
df = pd.read_csv('tiktok_dataset.csv')
df = df.dropna()

# Separate groups
verified = df[df['verified_status'] == 'verified']['video_view_count']
not_verified = df[df['verified_status'] == 'not verified']['video_view_count']

print("VERIFIED ACCOUNTS:")
print(f"  Count: {len(verified):,}")
print(f"  Mean: {verified.mean():,.2f}")
print(f"  Median: {verified.median():,.2f}")

print("\nNOT VERIFIED ACCOUNTS:")
print(f"  Count: {len(not_verified):,}")
print(f"  Mean: {not_verified.mean():,.2f}")
print(f"  Median: {not_verified.median():,.2f}")

In [ ]:
# Visualize
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='verified_status', y='video_view_count')
plt.yscale('log')
plt.title('Video View Count by Verification Status (Log Scale)')
plt.ylabel('View Count (Log Scale)')
plt.xlabel('Verification Status')
plt.show()

In [ ]:
# Two-sample t-test (Welch's)
t_stat, p_val = stats.ttest_ind(verified, not_verified, equal_var=False)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.10f}")

alpha = 0.05
if p_val < alpha:
    print(f"\nREJECT NULL HYPOTHESIS (p < {alpha})")
    print("There IS a statistically significant difference in mean view counts.")
else:
    print(f"\nFAIL TO REJECT NULL HYPOTHESIS (p >= {alpha})")

# Effect size (Cohen's d)
pooled_std = np.sqrt(((len(verified) - 1) * verified.std()**2 +
                      (len(not_verified) - 1) * not_verified.std()**2) /
                     (len(verified) + len(not_verified) - 2))
cohens_d = (not_verified.mean() - verified.mean()) / pooled_std
print(f"\nCohen's d: {cohens_d:.4f} (Large effect)")

print(f"\nKEY FINDING: Verified accounts get LOWER views on average!")
print(f"  Verified: {verified.mean():,.0f} avg views")
print(f"  Not verified: {not_verified.mean():,.0f} avg views")

---
## Course 4: Logistic Regression (PACE: Construct)

Practice model predicting `verified_status` before building the final claims classifier.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.utils import resample

# Reload clean data
df = pd.read_csv('tiktok_dataset.csv')
df = df.dropna()

# Feature engineering: text length
df['text_length'] = df['video_transcription_text'].str.len()
print(f"Mean text length: {df['text_length'].mean():.0f} characters")
print(f"Median text length: {df['text_length'].median():.0f} characters")

In [ ]:
# Handle class imbalance — upsample minority class
print("Original class distribution:")
print(df['verified_status'].value_counts())

majority = df[df['verified_status'] == 'not verified']
minority = df[df['verified_status'] == 'verified']

minority_upsampled = resample(minority,
                              replace=True,
                              n_samples=len(majority),
                              random_state=42)

df_balanced = pd.concat([majority, minority_upsampled])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced class distribution:")
print(df_balanced['verified_status'].value_counts())

In [ ]:
# Prepare features and target
feature_cols = ['video_view_count', 'video_like_count', 'video_share_count',
                'video_download_count', 'video_comment_count', 'text_length']

X = df_balanced[feature_cols]
y = (df_balanced['verified_status'] == 'verified').astype(int)

# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

# Evaluate
y_pred = log_reg.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Verified', 'Verified']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Verified', 'Verified'],
            yticklabels=['Not Verified', 'Verified'])
plt.title('Confusion Matrix: Verified Status Prediction')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (coefficients)
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': log_reg.coef_[0],
    'Abs_Coefficient': np.abs(log_reg.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Coefficient'])
plt.xlabel('Coefficient Value')
plt.title('Feature Importance in Logistic Regression Model')
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

print(feature_importance)

---
## Course 5: Final Claims Classification Model (PACE: Execute)

The primary project goal — classify videos as **claims** or **opinions** using Random Forest and XGBoost, optimized for **recall** (catching all claims is more important than avoiding false positives).

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score,
                             f1_score, ConfusionMatrixDisplay)

# Load and prepare data
data = pd.read_csv('tiktok_dataset.csv')
data = data.dropna()

# Feature engineering
data['text_length'] = data['video_transcription_text'].str.len()

print(f"Dataset: {data.shape}")
print(f"\nClaim status distribution:")
print(data['claim_status'].value_counts())

In [ ]:
# Text length distribution by claim status
plt.figure(figsize=(10, 6))
plt.hist(data[data['claim_status'] == 'claim']['text_length'],
         bins=30, alpha=0.6, label='Claim', color='darkblue', edgecolor='black')
plt.hist(data[data['claim_status'] == 'opinion']['text_length'],
         bins=30, alpha=0.6, label='Opinion', color='orange', edgecolor='black')
plt.xlabel('Text Length (characters)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of Video Transcription Text Length by Claim Status', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Prepare features
X = data.copy()
X = X.drop(['#', 'video_id', 'video_transcription_text'], axis=1)
X['claim_status'] = X['claim_status'].replace({'opinion': 0, 'claim': 1})
X = pd.get_dummies(X, columns=['verified_status', 'author_ban_status'], drop_first=True)

# Isolate target and features
y = X['claim_status']
X = X.drop('claim_status', axis=1)

print(f"Feature matrix: {X.shape}")
print(f"\nFeatures: {X.columns.tolist()}")

In [ ]:
# Train / Validation / Test split (60/20/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

print(f"Training set: {X_train.shape} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape} ({len(X_test)/len(X)*100:.1f}%)")

In [ ]:
# Random Forest with GridSearchCV (optimized for recall)
rf = RandomForestClassifier(random_state=42)

rf_cv_params = {
    'max_depth': [5, 7, None],
    'max_features': [0.3, 0.6],
    'max_samples': [0.7],
    'min_samples_leaf': [1, 2],
    'min_samples_split': [2, 3],
    'n_estimators': [75, 100, 200]
}

scoring = ['accuracy', 'precision', 'recall', 'f1']

rf_cv = GridSearchCV(rf, rf_cv_params, scoring=scoring, cv=5, refit='recall')

print("Training Random Forest with GridSearchCV...")
rf_cv.fit(X_train, y_train)
print(f"\nBest recall score: {rf_cv.best_score_:.4f}")
print(f"\nBest parameters:")
for param, value in rf_cv.best_params_.items():
    print(f"  {param}: {value}")

In [ ]:
# XGBoost with GridSearchCV (optimized for recall)
xgb = XGBClassifier(objective='binary:logistic', random_state=42, eval_metric='logloss')

xgb_cv_params = {
    'learning_rate': [0.1, 0.3],
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 3],
    'n_estimators': [75, 100, 200]
}

xgb_cv = GridSearchCV(xgb, xgb_cv_params, scoring=scoring, cv=5, refit='recall')

print("Training XGBoost with GridSearchCV...")
xgb_cv.fit(X_train, y_train)
print(f"\nBest recall score: {xgb_cv.best_score_:.4f}")
print(f"\nBest parameters:")
for param, value in xgb_cv.best_params_.items():
    print(f"  {param}: {value}")

In [ ]:
# Evaluate both models on validation set
rf_val_preds = rf_cv.best_estimator_.predict(X_val)
xgb_val_preds = xgb_cv.predict(X_val)

# Model comparison
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Random Forest': [
        accuracy_score(y_val, rf_val_preds),
        precision_score(y_val, rf_val_preds),
        recall_score(y_val, rf_val_preds),
        f1_score(y_val, rf_val_preds)
    ],
    'XGBoost': [
        accuracy_score(y_val, xgb_val_preds),
        precision_score(y_val, xgb_val_preds),
        recall_score(y_val, xgb_val_preds),
        f1_score(y_val, xgb_val_preds)
    ]
})

print("MODEL COMPARISON (Validation Set):")
print(comparison_df.to_string(index=False))
print("\nChampion Model: Random Forest (slightly better recall)")

In [ ]:
# Validation confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_val, rf_val_preds),
                       display_labels=['Opinion', 'Claim']).plot(cmap='Blues', ax=axes[0])
axes[0].set_title('Random Forest — Validation')

ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_val, xgb_val_preds),
                       display_labels=['Opinion', 'Claim']).plot(cmap='Greens', ax=axes[1])
axes[1].set_title('XGBoost — Validation')

plt.tight_layout()
plt.show()

In [ ]:
# Final evaluation on test set (Random Forest champion)
rf_test_preds = rf_cv.predict(X_test)

print("FINAL TEST SET EVALUATION — RANDOM FOREST (CHAMPION)")
print("=" * 55)
print(classification_report(y_test, rf_test_preds, target_names=['Opinion', 'Claim']))

# Confusion matrix
disp_test = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, rf_test_preds),
                                   display_labels=['Opinion', 'Claim'])
disp_test.plot(cmap='Blues')
plt.title('Random Forest — Final Test Set Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance analysis
importances = rf_cv.best_estimator_.feature_importances_

feature_importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values('importance', ascending=False)

print("Feature Importance Ranking:")
print(feature_importance_df.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['feature'], feature_importance_df['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance — Random Forest Claims Classifier')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## Summary & Recommendations

### Final Model Performance (Test Set)
| Metric | Score |
|--------|-------|
| Accuracy | ~99.5% |
| Precision | ~99.5% |
| Recall | ~99.5% |
| F1-Score | ~99.5% |

### Top Predictive Features
1. `video_view_count` — 61.3%
2. `video_like_count` — 23.4%
3. `video_share_count` — 9.3%

Engagement metrics account for 99.6% of predictive importance.

### Key Insights
- Claims get **100x more median views** than opinions (501,555 vs 4,953)
- Verified accounts get **lower** views on average (verification paradox)
- Banned authors get **50x more** engagement than active users
- Outliers (25–35% of data) are legitimate viral content — kept in model

### Business Recommendations
1. **Deploy model as initial content filter** with human moderator oversight
2. **Prioritize recall** — false negatives (missed claims) cause real harm
3. **Monitor performance weekly**, retrain quarterly
4. **Future enhancements:** NLP features, temporal patterns, creator-level features

### Ethical Considerations
- Human oversight required for all flagged content
- Regular bias auditing
- Transparency about model limitations
- Predictions should trigger support, not punitive action